# [BengioDLBackpropBrain] Yoshua Bengio. Deep learning & Backprop in the Brain

Read: 2025-11-24  
From: KrotovBilogLearnAlgo  
Source: https://www.youtube.com/watch?v=FhRW77rZUS8  

Слайд 6. Сетка обучалась различать МЕСТА. Но научилась различать объекты (люди, фонари и т.д.)

Слайд 7. RELU (biologicaly insipred) вместо сигмоид позволяет обучать более глубокие сетки (Glorot & Bengio, AISTATS 2011)  

Слайд 7. Зашумление (например, через spiking-like dropout) - это мощные регуляризаторы. Открывают путь для мощной генерализации

Слайд 11. Attention открывает путь к RW памяти. (я тоже так думал).

<img src="./img/2025-11-25/dl-backprop-1.jpg" width=500>

Слайд 16. Propagation of errors = propagation of surprises = getting back in harmony

Слайд 27. Mirror interneurons solve the linear feedback puzzle

Слайд 28. MNIST Experiments with mirror interneurons. Walter Sennn & Joao Sacramento

# [HintonCortexBackprop] Geoffery Hinton. Can sensory cortex do backpropagation?

Read: 2025-11-24  
From: KrotovBilogLearnAlgo  
Source: https://www.youtube.com/watch?v=cBLk5baHbZ8  

При желании можно сделать backprop и в биологической ткани. Аналогом supervised сигнала может выступать какое-то изменение входного сигнала, или предсказание следующего сигнала, или смесь сигнала с другой модальностью <= в этом плане созвучны мысли с [РдзбЛогСозн] (там контексты визуального слоя изучали правила преобразования следя за изменениями при саккадах).
Сам backprop может быть сделан на основе петель. По петлям транслируется то, что было предсказано и то, что ожидается. Симметричные веса/связи - необязательны! (см. Lillicrap, Cownden, Tweed & Akerman (2014)).
Чувствуется, что GLOM идеи [HintonGlomFinal] растут в том числе отсюда.

---

Four reasons why the brain cannot do backprop  
• There is no obvious source for the supervision signal.  
• Cortical neurons do not communicate real-valued activities: They send spikes.   
• The neurons need to send two different types of signal  
  - Forward pass output = activity = y  
  - Backward pass* output = error deriv w.r.t input = \$dE/dx\$   

• Neurons do not have symmetric reciprocal connections: The feedback connections do not even go back to the neurons that the feed-forward connections came from

**Sources of supervision** that allow backprop learning **without a separate supervision signal**   
• Reconstruct the all or part of the current frame.   
• Predict the next frame.   
• Make locally extracted features agree with features predicted from the local context or from another modality: "She scromed him with the frying pan"   

Why spikes are better than analog values  
• Synapses are much cheaper than training cases: We have **\$10^{14}\$** synapses and live for **\$10^9\$** seconds.  
• A good way to throw a lot of parameters at a task is to use big neural nets with dropout: Dropout has the same regularization advantages as averaging big ensembles of separate models but makes much more efficient use of hardware   
• Sending random spikes from a Poisson process is very similar to dropout: It is better than sending real values.   

An early use of the idea that temporal derivatives encode error derivatives (Hinton & McClelland, 1988)   
• Send activity round a closed loop of logistic units and see how the activity of a unit changes between one iteration and the next.   
• Modify the incoming weights of a unit to reduce this change. 

Combining STDP with reverse STDP (spike-time-dependent plasticity)  
a) First use reverse STDP to learn a stack of auto-encoders.  
b) Then do two top-down passes:  
- In the first pass, drive the units top-down from the top-level representation that was predicted from the input => The states of the units in the intermediate layers should stay roughly the same.  
- In the second pass, drive the units top-down from the desired top-level representation => The states of the units in the intermediate layers will change slightly.
  
c) Use the activity differences between the two passes as the postsynaptic term in the delta rule.  
- This corresponds to STDP because the correct  states come after the wrong states.  

What the two top-down passes achieve  
• The difference in the activities in the two top-down passes is an accurate approximation to the post-synaptic term in the backpropagation learning rule provided: a) The top-down weights are the transpose of the bottom-up weights (as they are in an RBM) , b) The auto-encoders are working well so that the first top-down pass does not change the activities much.  
• The means the top-level representation should be invertible (e.g. label + pose, not just label).  

<img src="./img/2025-11-25/cortex-backprop-1.jpg" width=500>


A problem  
• This way of performing backpropagation appears to require symmetric weights.   
• What happens if the top-down weights are not the same as the bottom-up weights?  
• In the extreme case what happens if we have sparse connectivity in both directions and we do not allow top-down connections between pairs of units that have bottom-up connections?  


Now a **miracle** occurs  
• Lillicrap, Cownden, Tweed & Akerman (2014) showed that backprop still works almost as well using fixed random top-down connection weights.  
— The bottom-up weights adapt so that the fixed top-down weights are approximately their pseudo-inverse near the data manifold   
• This result means that the stack of autoencoders does not need to have symmetric weights But it does need to be able to reconstruct well  


An Analogy   
• In **variational inference**, we do inference according to some simple scheme that is initially incorrect: The generative model then adapts to make our way of doing inference more correct.   
• In **feedback alignment**, the random feedback matrices initially give the wrong error derivatives for the activities in lower layers: The forward weights then adapt to make the error derivatives more correct (but why?)  


Summary of backpropagation in cortex  
• The fact that neurons send spikes rather than real numbers is not a problem: Spikes are a great regularizer.  
• Error derivatives can be represented as temporal derivatives: This allows a neuron to represent both its activity and its error derivative in the same axon.   
• Spike-time dependent plasticity is the signature of backpropagation learning using temporal derivatives as error derivatives.  
• The problem that each bottom-up connection needs to have a corresponding top-down connection is a non-problem.  


A consequence of using temporal derivatives to code error derivatives    
• The rate of change of the output of a neuron cannot be used to represent the rate of change of the quantity that the neuron represents: For example, the rate of change of the output
of a position sensitive neuron cannot be used to represent velocity.  
• Brain damage confirms this. If you lose velocity coding neurons, you see the position of a car change but you don't see it moving.  

# [KrotovBilogLearnAlgo] Dmitry Krotov. Unsupervised learning by competing hidden units

Read: 2025-11-24  
From: LiangFruitFlyWordEmb  
Source: https://www.pnas.org/doi/pdf/10.1073/pnas.1820458116  

Тут идеи, которые используются в [LiangFruitFlyWordEmb].
Без учителя обучаем зону (набор из 2000 нейронов) на выделение фичей (факторов). Для этого используется модифицированное правило Ойа (Oja) и ингибиционный механизм. В общих словах схема такая:
1) ингибиционный механизм роутит сигнал к тем нейронам, которые самые многообещающие. Эти нейроны обучаются по модифицированному правилу Ойа с положительное добавкой. Эти нейроны настраиваются на сигнал.
2) ингибиционные механизм роутит сигнал к нейронам, которые слабо на сигнал среагировалис. Эти нейроны обучаются по модифицированному правилу Ойа с отрицательной добавкой. Таким образом эти нейроны отгоняются от сигнала.
3) нейроны, которые не среагировали/отрицательно среагировали, не трогаются.

Отличие от правила Ойа в параметре \$R^p\$. Типа радиус сферы. Если R=1, и убрать фактор нормализации, то практически один в один правило Ойа. Короче, фильтра Хебба получается.

Если изучить ноутбук Unsupervised_learning_algorithm_MNIST, то пункты 1), 2) и 3) там ещё более упрощены. Выбирается один нейрон - победитель, которые получает добавку к весам. Также выбриается нейрон лузер, который теряет веса. Остальные стоят в сторонке.

Всё вместе это даёт, что зона выучивает различные факторы. Если бы не было ингибиционного механизма, то нейроны зоны выучили одинаковые факторы. А с ингибиционным механизмом мы заставляем нейроны обучаться разному. 

Очень похоже на то, что описано в [РдзбЛогМышл] в главе "Часть 2. Факторы". Там тоже рассматривались алгоритмы декорреляции (например, APEX, карты Кохонена), которые заставляют зону учиться разному. Если отвлечься от биологической правдоподбности, то можно использовать обобщённый фильтр Хебба, для которого требуется централизованное управление (надо передавать управление тому или иному нейрону по очереди).

В конце присобачиваем ещё один слой, который уже что-то полезное делает. Например, классифицирует MNIST. Этот слой уже можно end-to-end через backpropagation обучить. Reservoir computing [VrugtIntroReservComp]?

We now know that:
1) neurons make either excitatory or inhibitory outgoing synapses, but not both (18)
2) that for excitatory synapses the change in efficacy can either increase or decrease, depending on whether the coordinated activity of the postsynaptic cell is strong or weak [long-term potentiation (LTP) and long-term depression (LTD)] (19);
3) that homeostatic processes regulate overall synaptic strength;
4) that prepost timings of neuronal action potentials can be very important (STDP) (20);
5) that changes in synaptic efficacy may be quantized (21);
6) that there are somewhat different rules for changing the strength of excitatory and inhibitory synapses (22), etc

---

> Thus, the learning rule is **nonlocal**, i.e., requires information about the states of other specific neurons in the network in addition to the two neurons that are connected by the given synapse. In biology, the synapse update depends on the activities of the presynaptic cell and the postsynaptic cell and perhaps on some global variables such as how well the task was carried out (state of animal attention, arousal, fear, etc.), but not on the activities other specific neurons.

> End-to-end training with backpropagation can also be used to generate useful early representations in an unsupervised way. An autoencoder can be trained to carry out a self-mapping task on
input patterns

>  The idea of BCM theory is that for a random sequence of input patterns a **synapse** is learning to differentiate between those stimuli that excite the postsynaptic neuron strongly and those stimuli that excite that neuron weakly

Синапс - это first-class citizen? А не нейрон?

> From the AI point of view the main drawback of the presented algorithm is that it is **slow**. There are two reasons for this.  
> First, it is an online algorithm so that training examples are presented one at a time, unlike SGD where training examples can be presented in minibatches.  
> Second, for any training example one has to wait until the set of hidden units reaches a steady state. This requires numerically solving Eq. 8, which is time consuming.

Онлайн (пример за примером) алгоритмы медленные по сравнению с батчёвыми (batched). SGD типа батчёвый.

> Thus, the learned feature detectors encode both where the **ink is in the data** and where the **ink is not**.

Нейроны имеют и плюсовые и минусовые веса. Т.е. конрастность есть - нейрон что-то любит, а что-то нет.

<img src="./img/2025-11-24/unsupervised-learn-compete-1.jpg">
 
> When a network is presented with an input, **many hidden units are activated**. Activities of these hidden units are then used in the next layer to arrive at the classification decision. Thus, the network learns a **distributed representation** of the data over multiple hidden units.

Distributed representation!!! 

https://www.youtube.com/watch?v=4lY-oAY0aQU

https://web.archive.org/web/20201015150113/https://www.ibm.com/blogs/research/2019/04/biological-algorithm/

https://github.com/DimaKrotov/Biological_Learning



# [KaplanPoliticalBeliefsCounterevidence] Jonas Kaplan et al. Neural correlates of maintaining one’s political beliefs in the face of counterevidence

Read: 2025-11-23    
Source: https://www.nature.com/articles/srep39589  
File: Kaplan et al. Neural correlates of maintaining ones political beliefs in the face of counterevidence  

> Our results show that when people are confronted with challenges to their deeply held beliefs, they preferentially engage brain structures known to support stimulus-independent, internally directed cognition. Our data also support the role of emotion in belief persistence. Individual differences in persuasion were related to differences in activity within the insular cortex and the amygdala—structures crucial to emotion and feeling. The brain’s systems for emotion, which are purposed toward maintaining homeostatic integrity of the organism63,
appear also to be engaged when protecting the aspects of our mental lives with which we strongly identify, including our closely held beliefs.

---

Мы защищаемся, когда встречаемся с чем-то, что опровергает наши убеждения и верования. Цель защиты - удержание гоместазиса, чтобы не разрушить целостное "Я"

# [LiangFruitFlyWordEmb] Yuchen Liang. Can A Fruit Fly Learn Word Embeddings?

From: DurhamGPT-3Дома  
Read: 2025-11-23  
Source: https://arxiv.org/pdf/2101.06887  
File: Liang. CAN A FRUIT FLY LEARN WORD EMBEDDINGS  

<img src="./img/2025-11-23/fruit-fly-1.jpg" width=600>

> The network motif shown in Fig. 1 is believed to be responsible for clustering sensory stimuli so that similar stimuli elicit similar patterns of neural responses at the level of KCs to allow generalization, while distinct stimuli result in different neural responses, to allow discrimination. Importantly, this biological network has evolved to accomplish this task in a very efficient way.

> Although this network has evolved to process sensory stimuli from olfaction and other modalities and not to “understand” language it uses a general purpose algorithm to embed inputs (from different modalities) into a high dimensional space with several desirable properties, which we discuss below.

<img src="./img/2025-11-23/fruit-fly-2.jpg" width=600>

> Intuitively, the goal of the training algorithm is to adjust the weights of the neural network so that they are aligned with w-grams that are frequently present in the training corpus. We rely on the assumption that semantically related w-grams share several “core” words, while a few individual words might be substituted by synonyms/antonyms

> We show below that these **hash codes capture semantic meaning of the target word and the context** in which it is used.

<img src="./img/2025-11-23/fruit-fly-3.jpg" width=600>

>  ...while the goal of the present paper is to **learn correlations between two different “modalities”: target word and its context**

---

Интересно:
1) в работе рассматриваются бинарные разреженные вектора (bio hash coding) - binary word embeddings
2) Плюс сама сетка сравнительно простая.

Интересно было бы поковырять!!! Исходники здесь https://github.com/bhoov/flyvec

# [DurhamGPT-3Дома] 30 миллиардов параметров: реально ли обучить русский GPT-3 в «домашних» условиях?

Read: 2022-07-01  
Read: 2025-11-22  
Source: https://habr.com/ru/post/565564/  

> Например, биологический мозг не работает таким образом. Мозгу вовсе не нужно вычислять выходы каждого из 100 миллиардов нейронов для любого действия — он умеет задействовать только те части, которые реально нужны для конкретной задачи. Аналогично, если мы учим новое слово, это не значит, что будут обновлены "веса" всех нескольких триллионов синапсов.

> Эта мысль реализуется, например, в архитектуре **Mixture of Experts (MoE)**. Данных подход стал известен после публикации [5]. Суть заключается в том, что существует некий переключатель, который для каждого шага выбирает несколько «экспертов» т. е. частей модели со своим набором весов, что сокращает необходимое количество вычислительных операций (общий принцип упрощенно показан на рисунке 1).

> В 2020 году исследователи из Google Brain упростили этот подход, оставив для каждого шага только одного эксперта, что улучшило масштабируемость и сделало обучение более стабильным. Это дало возможность Google обучить Switch Transformer [6].

Большая модель представляется в виде набора более мелких. И далее есть переключалка (router). Как обычно Hinton приложил руку

<img src="./img/2025-11-22/gpt-3-home-1.jpg" width=600>

> Многие ученые считают, что правило обучения биологического нейрона должно быть **локальным** (т. е. зависеть только от непосредственного окружения нейрона, без передачи градиентов). Существует даже много попыток заменить обратное распространение ошибки на локальное правило [7, 8], однако на задаче построения моделей языка такие вещи пока работают не очень хорошо. Поэтому я пошел компромиссным путем — каждый эксперт обучается обычным образом (и представляет из себя обычный трансформер), а взаимодействовать эксперты учатся с помощью **локального правила**.

Локальное правило - например, Hebb правило. Интересно будет изучить современные локальные правила, которые хотят заменить backpropagation:  
https://arxiv.org/pdf/2101.06887 - CAN A FRUIT FLY LEARN WORD EMBEDDINGS?
https://arxiv.org/pdf/2405.15868 - LLS: Local Learning Rule for Deep Neural Networks Inspired by Neural Activity Synchronization  
https://ojs.aaai.org/index.php/AAAI/article/view/26118/25890 - Backpropagation-Free Deep Learning with Recursive Local Representation Alignment  

# [BengioNeuralProbLangModel] Yoshua Bengio et al. A Neural Probabilistic Language Model

From: ColahDLNLPRepr  
Date: 2025-11-21  
Source: https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf  
File: Bengio.A Neural Probabilistic Language Model  

> For example, if one wants to model the joint distribution of 10 consecutive words in a natural language with a vocabulary V of size 100,000, there are potentially \$100000^{10} − 1 = 10^{50} − 1\$ free parameters. ... For discrete spaces, the generalization structure is not as obvious: any change of these discrete variables may have a drastic impact on the value of the function to be estimated, and when the number of values that each discrete variable can take is large, most observed objects are almost maximally far from each other in hamming distance.

Проклятие размерности

> In high dimensions, it is crucial to distribute probability mass where it **matters** rather than uniformly in all directions around each training point.

Т.е. надо распространять в нужные места (направления), а не просто во все одинаково.

>  Thus, n-gram models construct tables of conditional probabilities for the next word, for each one of a large number of contexts, i.e. combinations of the last n−1 words: \$\hat{P}(w_t|w_1^{t−1} ) ≈ \hat{P}(w_t|w_{t-n+1}^{t-1})\$.

Эргодичная цепь Маркова? В примерах TriadicMemory [OvrmnDTMem] делалось тоже самое. 

> What happens when a new combination of n words appears that was not seen in the training corpus?

Обычное дело - out-of-vocabulary случай.

<img src="./img/2025-11-21/neural-prob-lang-model-1.jpg" width=600>

Ключевая идея. Главное - тренировать **одновременно** и эмебиддинги и целевую функцию

> Why does it work? In the previous example, if we knew that `dog` and `cat` played similar roles (semantically and syntactically), and similarly for `(the,a)`, `(bedroom,room)`, `(is,was)`, `(running,walking)`, we could naturally generalize (i.e. transfer probability mass) from  
> `The cat is walking in the bedroom`  
> to `A dog was running in a room`  
> and likewise to `The cat is running in a room`  
> `A dog is walking in a bedroom`  
> `The dog was walking in the room`
> ...
> 
> Therefore, the presence of only one of the above sentences in the training data will increase the probability, not only of that sentence, but also of its combinatorial number of “neighbors” in sentence space (as represented by sequences of feature vectors).
>
> Experiments suggest that learning jointly the representation (word features) and the model is very useful.

Эмбеддинг должен не только свойства явления должен описывать, но и также свойства которые позволяют из него делать articulated strucutres (см. [HintonLearnDistrRepr]). 

<img src="./img/2025-11-21/neural-prob-lang-model-2.jpg" width=600>

Видны residual connections от элементов \$C(w_i)\$ к верхнему слою, где \$softmax\$. Фишка архитектуры трансформеров ([TransCircMathFrmwrkForTransformer])!

> Sometimes, part of an update on the parameter vector by one of the processors is lost, being overwritten by the update of another processor, and this introduces a bit of noise in the parameter updates. However, this noise seems to be very small and did not apparently slow down training.

Можно даже без блокировок обучать сеть в распрделенном режиме. Даже перезаписывания весов не страшны.

<img src="./img/2025-11-21/neural-prob-lang-model-3.jpg" width=600>

Разумная идея с N-gram моделями. Если есть специфичная инфа, то берём её. Если нет, то берём более общую и т.д.

> **Out-of-vocabulary words**. An advantage of this architecture over the previous one is that it can easily deal with out-of-vocabulary words (and even assign them a probability!). The main idea is to
first guess an initial feature vector for such a word, by taking a weighted convex combination of the feature vectors of other words that could have occurred in the same context, with weights proportional to their conditional probability

ВОТ ЭТО ПРЯМ БОМБА! Это то, чего я так долго искал. Ответ на вопрос: как выводить новое знанине. Мы же тоже можем догадаться примерно о значении слова, даже если видим его впервые. Вот оно как делается. Кстати в [ColahDLNLPRepr] тоже был такой пример: "Even though I’ve never seen a Aesculapian snake or an Armadillo before, if you show me a picture of one and a picture of the other, I can tell you which is which because I have a general idea of what sort of animal is associated with each word. These networks can accomplish the same thing."

> ... proposed approach allows to take advantage of the learned distributed representation to **fight the curse of dimensionality with its own weapons**: each training sentence informs the model about a combinatorial number of other sentences.

---

В целом логичное развитие и масштабирование идей из [HintonLearnDistrRepr]. Основной инсайт про out-of-vocabulary слова - эмбеддинги могут и эту проблему решать!!!

# [МацкевичBehavioralEnineering] Дмитрий Мацкевич. Школа менеджмента — Behavioral engineering

From: RandomSurf  
Date: 2021-01-19  
Date: 2025-11-20  
Source: https://www.youtube.com/watch?v=70hcbjyJZ-Q  

> ... начать это понимание что как бы вы не напрягались вы все равно забудете 80 процентов контента уже через час. так устроен мозг, поэтому сильно напрягаться не надо. лучше расслабиться. и задачи у этой лекции нет, чтобы там за этот час вы там прямо научились чему-то новому. это в принципе невозможно. задача сделать так чтобы вы заинтересовались и поняли, что вот здесь вот в этом направлении есть что-то увлекательность какая-то глубина, и вас появились фильтры восприятия мы начали это замечать своей работе вы начали где-то там читать об этом и так далее и по сути моя задача запустить вот  этот дофаминовые цикл

> депрессия это когда вы учите замечать негативное во всем, чтобы с вами не происходило

> потому что task manager обычно дарит одну лишь боль вы смотрите вечером или с утра и видите количество task-ов которые вы не сделали ты блин я просто ненавижу себя. смотрит на следующий день там этих task-ов еще больше и вы думаете были наверно это какой то просто плохой продукт я лучше как-нибудь сам без него там на бумажке в уме у меня общем это субъективно лучше получается хотя объективно это полезный продукт он утилитарный он вы должны использовать

> большая часть проектов с которыми я сталкиваюсь это как раз проекты где надо разгадать секрет о людях.  не секрет природы. секрет природы например как нам сделать ракету которая в 10 раз дешевле или как нам ну например сделать машину которая сама по себе ездит 
> ... но по большей части это секреты которые люди сами не знают о себе. то есть они не знают чего они хотят

> джон гурман который занимается психология взаимоотношений. он очень много развлекался опять же семейными парами когда он спрашивал по отдельности а в каком проценте случаев например вот ты выносишь мусор или там в каком проценте случаев именно ты являешься инициатором. потому что у людей очень много искажения реальности даже если я видел все случаи когда мой партнер выносит мусор когда я сам выношу мусор для меня это реально эмоциональное событие он пахнет как бы я напрягаюсь когда я вижу как выносить мусор то другой ну типа окей. но в моем мире я трудился сильно больше и все

> когда вы что-то новое пытаетесь придумать у вас мозг перемещается между двумя так называемыми условно там стадиями фокус diffusion соответственно и на фокус. когда мы сфокусированы что-то поглощаем мы не можем придумать ничего нового мы можем лишь что-то понимать там читать правила, там примеры решения и так далее. можно придумать что-то по-настоящему новое когда мы находимся в diffusion, но когда мы находимся в give you stay что когда мы думаем о самой проблеме мы не знаю там курим где-то китаем ст велосипеде принимаем душ к нам приходит решение проблемы но приходит но только лишь тогда когда у нас в голове чанки так называемые с этой с кем-то кусочками информации у которых есть контекст применению и через этот контекст применения у нас какая-то  проблема она коннектится с кучей разных кусочков которые по сути привод к одному решению. в сфокусированном состоянии мозг не может удержать одновременно 2 3 взаимосвязи бесполезно пытаться что-то придуматью это как разница между не знаю человеком которого заставляют учить физику он знает все формулы но не может участок в олимпиадах по физике потому что он просто заучила может решать только примеры которые он сфокусирована помнит как решать. а если 
надо чтобы человек побеждал в олимпиадах по физике он должен искренне любить этот предмет у него для каждого закона физики должно быть очень много контекстов применения которых он переосмыслил нашел очень много применение пришел к этому сам он закрывал глаза представлял что на электрон летит по кристаллической решетки переживал ночами не спал что-то всем происходит когда он сталкивается ему после этого не надо знать закон ома он его в принципе понимает он может его вывести на ощущение на интуиция и интуитивно использовать его когда он генерит какое-то решение неизвестной задачи соответственно чтобы вы могли решать неизвестные задача не двигаться просто шаблонно бесполезно пытаться там что-то учить запоминать надо это понимать понимать то есть понимать фундаментальный принцип почему оно именно такое какое есть и дальше максимизировать количество связей с этим там чанком информации

> советую это контекст применению то есть попытались найти в работе попытались это кому-то рассказать попытались об этом написать завод презентацию доклад а именно как бы поэтому я здесь выступаю потому что как бы каждый раз когда я выступаю я у себя сильнее сильнее укладываю это в голове я то что не занимаюсь благотворительностью или каким-то альтруизмом соответственно

> каждый раз когда мы тратим энергию и мозг все-таки перестает лениться он испытывают боль потому что это угроза угрожает его выживанию то есть каждый кто заставляет меня думать я его ненавижу. да то есть я на рационализируем как-то например там вы пришли на встречу с инвестором и показали ему стол умных слайдов в как в каждом из которых надо разбираться глубоко 3 к тридцатому слайду он поймет что ничего больше не понимает и мне будет два выбора подумать что он дурак либо вы дурак да то есть да тут не надо знать про brain сайт чтобы понять что он выберет потому что есть он подумает что он дурак мозг тоже не дурак он понимает что если я вдруг дурак то у меня возникает как бы  задача стать умным то есть это реально как бы нам карнитин и диссонанса который надо избавляться заниматься психотерапевтом потратить кучу денег усилий намного проще сказать что ну я все тот же как бы умный чувак и успешный  ну вот он какой то crazy санте ст который ну городец какую-то фигню

> и обязательно надо каждый раз когда вы читаете эту книгу чтобы она не была вестит надо обязательно кому-то рассказывать желательно человеку который вообще ничего не понимают не в продуктах не в brains ничего можно там своей маме либо случайному человеку на улице потому что это позволяет вам пересобрать эти воспоминания и укрепить их если вы просто что-то перечитываете это но практически не работает

<img src="./img/2025-11-20/beh-eng-0.jpg" width=600> <img src="./img/2025-11-20/beh-eng-1.jpg" width=600> <img src="./img/2025-11-20/beh-eng-2.jpg" width=600>
<img src="./img/2025-11-20/beh-eng-3.jpg" width=600> <img src="./img/2025-11-20/beh-eng-4.jpg" width=600>

# [HintonLearnDistrRepr] Geoffrey Hinton. Learning Distributed Representations Of Concepts

Read: 2025-11-20  
From: ColahDLNLPRepr  
Source: https://www.cs.toronto.edu/~hinton/absps/families.pdf  
File: Hinton. LEARNING DISTRIBUTED REPRESENTATIONS OF CONCEPTS  

>  This paper shows how the network can be made to **choose the pattems itself** when shown a set of propositions that use the concepts. It **chooses patterns which make explicit** the underlying features that are only **implicit** in the propositions it is shown.

> These range from extreme localist theories in which each concept is epresented by a single neural unit to extreme distributed theories in which a concept corresponds to a pattern of activity over a large part of the cortex.  
> These two extremes are the natural implementations of two different theories of semantics.  
> In the **structuralist approach**, concepts are defined by their relationships to other concepts rather than by some internal essence. The natural expression of this approach in a neural net is to make each concept be a single unit with no internal structure and to use the connections between units to encode the relationships between concepts.  
> In the **componential approach** each concept is simply a set of features and so a neural net can be made to implement a set of concepts by assigning a unit to each feature.

Две крайности, про которые я тоже думал. Либо ничего не значающий код (рандомный) для обозначения сущности и есть связи. Либо для описания сущности используем код, в котором отдельные разряды описывают те или иные фичи.

Тут Хинтон говорит следующее. В структурном подходе фокус делается на взаимосвязях. Значит, можно научиться собирать высокоуровневые структуры из сущностей (например, из слов - предложения). Но при этом нет возможности проводить аналогии, искать похожести и т.д.

Напротив, в компонентом подходе можно проводить аналогии, искать похожести. Но непонятно, как собирать высокоуровневые структуры ("no obvious way of representing articulated structures composed of a number of concepts"). 

Поэтому Хинтон говорит, что нужно такое представление сущностей, которые бы сочетало и то, и другое. Т.е. и взаимосвязи (и роли) описывало бы, так и свойства сущностей.

>  Each unit then represents the conjunction of a role with a feature of the concept filling that role.  It has the interesting property that the very same concept will have quite different representations when it is playing different roles.

Эмбеддинги и их мутация?

> The use of random patterns ... entirely eliminates one of the most interesting aspects of distributed representations: **The ability to capture the similarity between concepts by the similarity of their representations**, and the consequent **ability to generalize new information in sensible ways**.

Рандомные вектора не позволяют обобщать. Чувствую, что тоже в это упёрся, когда ковырял РдзбКонтексты / HDC / TriadicMemory. 

> Micro-inferences store propositions by encoding the underlying regularities of a domain. ...  If the network has leamed the micro-inference given above it will have a natural tendency to make sensible
guesses. If, for example, it is told enough about a new person, Jane, to know that Jane is old and it is then asked to complete the proposition (Jane has-husband ?) it will expect the filler of the person2 role to be old.

Захват и описание регулярностей предметной области.

<img src="./img/2025-11-20/learn-distr-repr-1.jpg" width=600>

> The gradient descent procedure used by the network also has its analog in scientific research.
>
> But the back-propagation phase is central to the learning procedure and it is quite **unlike anything known to occur in the brain**. The connections are all used backwards, and the units use a different input-output function. We therefore view this learning procedure as an interesting way of demonstrating what can be achieved by gradient descent, **without claiming that this is how gradient descent is actually implemented in the brain**.

Хинтон признаёт, что backpropagation не имеет аналогов в мозгу.

---
<img src="./img/2025-11-20/learn-distr-repr-2.jpg" width=600>

**Поразительно!**
1) Обучаем сетку на датасете, описывающем картинку, т.е. на троицах вида:
   "Christopher-husband-Penelope"  
   "Christopher-wife-Penelope"  
   "Margharet-daughter-Christopher"  
   "Margharet-daughter-Penelope"  
   ...

2) Получаем набор эмбеддингов, в которых можно узнать признаки "English/Italian", "Возраст", "Ветвь родства" т.д. (всего 6 свойств в работе было описано)

Т.е. получаем признаки, которых не было в датасете в явном образе!

---

Почему *distributed* representation? Потому что мы от OHE (один разряд) переходим к многомерному представлению. Этакий explosion делаем

---

Даже сетка относительно простой структуры позволяет выучить эмбиддинги, которые передают смысл и которые позволяют делать обобщения. Т.е. все как в [ColahDLNLPRepr] "You’ve seen all the words that you understand before, but you haven’t seen all the sentences that you understand before. So too with neural networks.".

Кажется, что это краеугольный камень, это кровь мыслительного процесса.

# [ColahDLNLPRepr] Colah. Deep Learning, NLP, and Representations

Read: 2025-11-20  
From: TransCircToyModelSuperposition  
Source: https://colah.github.io/posts/2014-07-NLP-RNNs-Representations/#word-embeddings  
File: colah. Deep Learning, NLP, and Representations.pdf  

> I’d like to start by tracing a particularly interesting strand of deep learning research: word embeddings. ... I think they are one of the best places to gain intuition about why deep learning is so effective.

> **You’ve seen all the words that you understand before, but you haven’t seen all the sentences that you understand before. So too with neural networks.**

Это то, что о чём я думал и искал долго. Что за механизм, которые позволяет решить проблему (все слова известны, но все предложения невозможно изучить. Но как-то мы понимаем предложения же).

> It’s important to appreciate that all of these properties of W are __side effects__. We didn’t try to have similar words be close together. We didn’t try to have analogies encoded with difference
> vectors. All we tried to do was perform a simple task, like predicting whether a sentence was valid. __These properties more or less popped out of the optimization process__.>

> This general tactic – learning a good representation on a task A and then using it on a task B – is one of the major tricks in the Deep Learning toolbox. It goes by different names depending on the details: **pretraining**, transfer learning, and multi-task learning.

Вот откуда G**P**T

> Of course, we observe that the words we knew had similar meanings end up close together. Since we optimized for that, it’s not surprising. **More interesting is that words we didn’t know were translations end up close together**.

Решение проблемы out-of-vocabulary

> Intuitively, it feels a bit like the two languages have a similar **‘shape’ and that by forcing them to line up at different points, they overlap and other points get pulled into the right positions**.

т.е. потянув только за некоторые ниточки, мы перетягиваем всё полотно смысла целиком и согласованно, т.к. у предметной области есть структура. Если бы структуры не было (аморфное тело, вода), то этот трюк бы не сработал!

---

Получается что эмбеддинги (distributed representation) описывают (передают) СТРУКТУРУ знаний. И они позволяют делать обобщения и reasonable guesses.

# [WelchLabsDeepseekRewroteTransf] Welch Labs. How DeepSeek Rewrote the Transformer [MLA]

Read: 2025-11-19  
Source: https://www.youtube.com/watch?v=0VLAoVGf_74  
From: RandomSurf  

Куча памяти и вычисления приходится на Multi Head Attention. Transformer есть авторегрессивная машина - потребляет слово за словом, при этом то что, сам воспроизвел себе и скармливает (автоиндуктивная машина). Поэтому большая часть контекста не меняется. Поэтому можно KQ матрицы и V матрица не вычислять с нуля, а инкрементально обновлять (буквально последюнюю строку). Т.е. кешировать матрицы можно. Но в лоб это требует много памяти. Ребяти из Deepseek разработали латентное представление этих матрица, что позволяет и память, и вычисляния сэкономить, и эффективность модели сохранить, а то и повысить.

<img src="./img/2025-11-19/deepseek-1.jpg" width=500> <img src="./img/2025-11-19/deepseek-2.jpg" width=500>

# [3BL1BRAIImagesVideos] 3BL1BR. But how do AI images and videos actually work?

Read: 2025-11-19  
Source: https://www.youtube.com/watch?v=iv-5mZ_9CPY  
From:  3BL1BRLLMStoreFacts  

CLIP - Contrast Language Image Pre-training. Есть эмбединг для картинки, есть эмбидинг для текста. Составляем матрицу. Тренируем модель, которая максимизирует диагональные элементы и минимизирует нидиагональные. Получаем хороший классификатор картинок.  
https://arxiv.org/pdf/2103.00020  
<img src="./img/2025-11-19/ai-images-videos-1.jpg" width=500> <img src="./img/2025-11-19/ai-images-videos-2.jpg" width=500>

DDPM - Denosing Diffuson Probabilistic Models. Берем картинку, зашумляем её. Тренируем модель, чтобы она научилась вычислять этот шум. При сэмплинге итеративно очищаем картинку, но при этом добавляем другой шум. Без этого другого шума мы получил безликую, усредненную картинку.
https://arxiv.org/pdf/2006.11239  

<img src="./img/2025-11-19/ai-images-videos-3.jpg" width=500> <img src="./img/2025-11-19/ai-images-videos-4.jpg" width=500>

Теперь соединяем DDPM и CLIP. Что-то типа кондишнига получается.

<img src="./img/2025-11-19/ai-images-videos-5.jpg" width=500> <img src="./img/2025-11-19/ai-images-videos-6.jpg" width=500>


# [VrugtIntroReservComp] Michael te Vrugt. An introduction to reservoir computing

From: OvrmnDTMem
Read: 2025-11-14
Source: https://arxiv.org/abs/2412.13212

Есть система с высокой размерностью (пусть даже ведро с водой). К этой системе подключаем Feed-forward Neural Net (FNN). И эта FNN уже подвергается тренировка, а сам резервуар ведёт себя как есть. Свойство резервуара по разному реагировать на входные данные эквивалентно, что входные данные переводятся в пространство высокой размерности, где их классификаций/разделение существенно упрощаются. И поэтому FNN тренируется достаточно прямолинейно.

Рассматриваются системы, эволюция которых может быть описана рекурсивным уравнением \$\mathbf{X}(t_i) = \mathbf{f}(\mathbf{X}(t_{i-1}), \mathbf{u}(t_i))\$

Требуемые свойства резервуара:
- воспроизводимость. Чтобы на одни и те же данные был боль-мень схожий отклик
- разделимость. Разные сигналы должны приводить к разным откликам резевуара
- угасание. Отклик резервуара должен угасать по времени. Т.е. с одной стороны новые данные должны превалировать над историей, но и это не должно быть многовенным откликом (наличие памяти).

# [OvrmnDTMem] Peter Overmann. Triadic Memory - A Fundamental Algorigthm for Cognitive Computing

Описывается diadic/triadic memory. Такая память - суть комбинаторное хранилище, которое строится на попарной корреляции битов бинарных векторов. Сильно напоминает матрицу ковариации в МНР.

На основе Triadic memory автор показывает как построить более сложные механизмы, типа, автоэнкодера.

# [РдзбЛогСозн] Алексей Редозубов. Логика сознания

Source: 
- https://habr.com/ru/articles/308268/
- https://habr.com/ru/articles/308370/
- https://habr.com/ru/articles/308878/
- https://habr.com/ru/articles/308972/
- https://habr.com/ru/articles/309366/
- https://habr.com/ru/articles/309626/
- https://habr.com/ru/articles/310214/
- https://habr.com/ru/articles/310960/
- https://habr.com/ru/articles/312740/
- https://habr.com/ru/articles/317712/
- https://habr.com/ru/articles/320866/
- https://habr.com/ru/articles/321256/
- https://habr.com/ru/articles/326334/
- https://habr.com/ru/articles/551644/

File: Редозубов А. Логика сознания.docx

> Когда компьютеры были слабыми или, вообще не дай бог, приходилось считать вручную, очень важно было использовать такие методы, которые позволяли бы минимальными вычислениями получать хорошие результаты. Многие идеи «хороших» алгоритмов исходят именно из такой экономии вычислений.  
> Идеология нейронов-детекторов, обученных на размыто-усредненный образ, — это во многом дань «экономным» методам вычислений. Правда, само обучение далеко не всегда можно назвать экономным и быстрым, но зато результат – это сеть с относительно небольшим количеством нейронов.


Схожая мысль у [RichSuttonBitterLesson]

Про бинарные коды см. [TransCircToyModelSuperposition]. Бинарные коды в виде суперпозиций ествественно возникают в нейронных сетях и позволяет им кодировать гораздо больше понятий (фичей), чем позволяет размерность.

См. [HintonCortexBackprop] - идеи про самообучение зоны и про источником supervised сигнала.

# [РдзбЛогМышл] Алексей Редозубов. Логика мышления

Source:
- https://habr.com/ru/articles/214109/
- https://habr.com/ru/articles/214241/
- https://habr.com/ru/articles/214317/
- https://habr.com/ru/articles/214525/
- https://habr.com/ru/articles/214663/
- https://habr.com/ru/articles/214797/
- https://habr.com/ru/articles/215023/
- https://habr.com/ru/articles/215151/
- https://habr.com/ru/articles/215283/
- https://habr.com/ru/articles/215287/
- https://habr.com/ru/articles/215701/
- https://habr.com/ru/articles/216263/
- https://habr.com/ru/articles/216301/
- https://habr.com/ru/articles/216409/
- https://habr.com/ru/articles/216633/
- https://habr.com/ru/articles/216825/
- https://habr.com/ru/articles/217055/
- https://habr.com/ru/articles/217205/

File: Редозубов. Логика мышления.docx

# [3BL1BRAttentionTransf] 3BL1BR. Attention in transformers, step-by-step | Deep Learning Chapter 6

Read: 2025-11-16  
Source: https://www.youtube.com/watch?v=eMlx5fFNoYc  

По ходу прохождения через тракт трансформера эмбеддинг мутирует (см. [MathFrmwrkForTransformer]). В конце он может превратиться в какой-то другой токен, чем в начале. Мутирование эмебиддинга можно представить как небольшие добавки к компонентам вектора. Эти добавки дают attention heads.

> The aim of a transformer is to progressively adjust these embeddings so that they don't merely encode an individual word, but instead they bake in some much, much richer contextual meaning.

<img src="./img/2025-11-17/3bl1br-attention-1.jpg" width=500 /><img src="./img/2025-11-17/3bl1br-attention-2.jpg" width=500 />

Query - это типа как "Есть ли кто-из прилагательных впереди меня"? Key - вычисление в т.ч. насколько ты прилагательное. Далее матрица KQ, где значение в ячейке - насолько ты подходишь под Q. По факту обычное скалярное произведение. Дальше вычисляется значение (Value) - взвешеннее среднее по колонке матрица KQ. 

<img src="./img/2025-11-17/3bl1br-attention-3.jpg" width=500 /><img src="./img/2025-11-17/3bl1br-attention-4.jpg" width=500 /><img src="./img/2025-11-17/3bl1br-attention-6.jpg" width=500 />

# [3BL1BRLLMStoreFacts] 3BL1BR. How might LLMs store facts | Deep Learning Chapter 7

From: 3BL1BRAttentionTransf  
Read: 2025-11-17  
Source: https://www.youtube.com/watch?v=9-Jl0dxWQs8  

- Факты хранятся в MLP слое.
- В GPT-3 49152 строки в матрице MLP (типа 49152 вопроса)
- В high-dimension пространстве появляется интересное свойство (Jhonson-Lindenstrauss lemma), что есть много около-ортогональных векторов (89-91 градус). В 3-х мерном такую фишу не провернуть, нужно много измерений. На этом основано запихивание большого кол-ва фактов/фичей в ограниченный размер.
- каждый нейорон в MLP слое представляет сразу несколько фичей из-а свойства указанного выше. Типа OHE-hack
- короче предудыщие два - это суперпозиция



> After the sequence of vectors has flowed through many, many different iterations of both of these blocks, by the end, the hope is that each vector has soaked up enough information, both from the context, all of the other words in the input, and also from the general knowledge that was baked into the model weights through training,

<img src="./img/2025-11-18/llm-store-facts-1.jpg" width=600><img src="./img/2025-11-18/llm-store-facts-2.jpg" width=600><img src="./img/2025-11-18/llm-store-facts-3.jpg" width=600>

# [RichSuttonOnTheLLM] Richard Sutton. Father of RL thinks LLMs are a dead end

Source: https://www.youtube.com/watch?v=21EYKqUsPfg  
Read: 2025-11-18

Ричард высказал мысль, что дескать LLM всё равно обучаются с учителем, тогда как в природе нет обучения с учителем. Типа настоящий интеллект либо наблюдает за происходящим, либо что-то делает и смотрит на результат. Сначала я не понял - ведь как таковой разметки же нет, а раз нет разметки, то и нет обучения с учителем. А потом понял, что разметка есть - это сам текст. Т.е. есть текст, есть следующее слово. Это и есть ground truth. Просто здесь нет такого процесса, как разметка данных, когда куча людей сидит и помечает: "Это есть то-то", "А это есть это". Сама разметка и есть текст. 

Про неибежность AGI и про будещее человеков
> I do think succession to digital intelligence or augmented humans is inevitable. I have a four-part argument.  
> Step one is,  there's no government or organization  that gives humanity a unified point of view that dominates and that can arrange... There's no consensus about how the world  
should be run.  
> Number two,  we will figure out how intelligence works. The researchers will figure it out eventually.  
> Number three, we won't stop just  with human-level intelligence. We  will reach superintelligence.  
> Number four, it's  inevitable over time that the most intelligent things around would gain resources and power. Put all that together and it's sort of inevitable.   
> You're going to have succession to AI  or to AI-enabled, augmented humans. Those four things seem clear and sure to happen. But within that set of possibilities,  there could be good outcomes as well  as less good outcomes, bad outcomes.   
> I'm just trying to be realistic about where  we are and ask how we should feel about it. I agree with all four of those  arguments and the implication.  I also agree that succession contains a wide variety of possible futures. Curious to get more thoughts on that. I do encourage people to   think positively about it. First of all, it's something we humans have  
always tried to do for thousands of years, try  to understand ourselves, trying to make ourselves   think better, just understanding ourselves. This is a great success for science, humanities.  
> We're finding out what this essential part of  humanness is, what it means to be intelligent. Then what I usually say is  that this is all human-centric.  But if we step aside from being a human and  just take the point of view of the universe,  this is I think a major stage in the universe, a  major transition, a transition from replicators. 
> We humans and animals,  plants, we're all replicators.  That gives us some strengths and some limitations. We're entering the age of design  because our AIs are designed. Our physical objects are designed, our buildings  are designed, our technology is designed. We're designing AIs now, things that can be intelligent themselves and that  are themselves capable of design.  This is a key step in the  world and in the universe. It's the transition from the  world in which most of the   interesting things that are, are replicated. Replicated means you can make copies of them,  but you don't really understand them. Right now we can make more intelligent beings,  more children, but we don't really  understand how intelligence works. Whereas we're reaching now to  having designed intelligence,   intelligence that we do understand how it works. Therefore we can change it in different  ways and at different speeds than otherwise. In our future, they may not be replicated at all.  
> We may just design AIs, and those  AIs will design other AIs, and everything will be done by design and  construction rather than by replication. I mark this as one of the four  great stages of the universe.  First there's dust, it ends with stars. Stars  make planets. The planets can give rise to life. Now we're giving rise to designed entities. I think we should be proud that we are giving  rise to this great transition in the universe.  It's an interesting thing. Should we consider them part of humanity or different from humanity? It's  our choice. It's our choice whether we should say,   "Oh, they are our offspring and we should  be proud of them and we should celebrate their achievements."Or we could say, "Oh no,  they're not us and we should be horrified."  It's interesting that it  feels to me like a choice. Yet it's such a strongly held thing  that, how could it be a choice?  I like these sort of contradictory  implications of thought.   

# [RichSuttonBittLesson] Richard Sutton. The Bittern Lesson

Read: 2025-11-18  
From: RichSuttonOnTheLLM  
Source: http://www.incompleteideas.net/IncIdeas/BitterLesson.html  

В перспективе побеждают методы, которые масштабируются и бенефитят от всё более доступных вычислительных ресурсов. Пытаться построить AI подобно тому, как наш мозг устроен - бесперспективно. См. также [РдзбЛогСозн]

> One thing that should be learned from the bitter lesson is the great power of general purpose methods, of methods that continue to scale with increased computation even as the available computation becomes very great. The two methods that seem to scale arbitrarily in this way are search and learning.
>
> We have to learn the bitter lesson that building in how we think we think does not work in the long run. The bitter lesson is based on the historical observations that 1) AI researchers have often tried to build knowledge into their agents, 2) this always helps in the short term, and is personally satisfying to the researcher, but 3) in the long run it plateaus and even inhibits further progress, and 4) breakthrough progress eventually arrives by an opposing approach based on scaling computation by search and learning. The eventual success is tinged with bitterness, and often incompletely digested, because it is success over a favored, human-centric approach.
>
> The second general point to be learned from the bitter lesson is that the actual contents of minds are tremendously, irredeemably complex; we should stop trying to find simple ways to think about the contents of minds, such as simple ways to think about space, objects, multiple agents, or symmetries. All these are part of the arbitrary, intrinsically-complex, outside world. They are not what should be built in, as their complexity is endless; instead we should build in only the meta-methods that can find and capture this arbitrary complexity. Essential to these methods is that they can find good approximations, but the search for them should be by our methods, not by us. We want AI agents that can discover like we can, not which contain what we have discovered. Building in our discoveries only makes it harder to see how the discovering process can be done.


# [VaswaniAttentionIsAllYNeed] Ashish Vaswany. Attention Is All You Need

Read: 2022-12-01  
Read: 2025-11-16  

Описание устройства трансформера с attention. RNN очень медленные из-за своей последовательной природы, параллелизация невозможна. Трансформер решает эту проблему RNN. 

Ключевые моменты устройства:
- positional encoding, где используется синусоиды различных частот для кодирования позиции токенов в контексте
- encoder из 6-ти слоёв. Каждый слой содержит 8 multi-head self-attention
- decoder из 6-слоёв. Струкурно похож на encoder, только есть ещё cross-attention
- residual connections. Роль этих connections описана в [TransCircMathFrmwrkForTransformer]. По факту - это основой тракт информации, когда attention heads "наливают" свой контекнт

Self-attention параллелизуется и позволяет выучивать long-range зависимости.

![](./img/2025-11-17/attention-all-you-need-1.jpg)

Архитектура трансформера благодаря вот этим attention heads даёт нейросети возможность выразить высказывания "Мне важно то-то. Я завишу от того-то. Если есть то, то надо изменить так-то"

# [TransCircToyModelSuperposition] Transformer Circuits. Toy Models of Superposition

Read: 2025-11-19  
Source: https://transformer-circuits.pub/2022/toy_model/index.html  
File: toy_models_of_superposition.mht  
From: 3BL1BRLLMStoreFacts  

Пачку нейронов можно представить как n-мерный вектор. Активность одного нейрона = ненулевое значение в соотв. разряде. Если нейрон соответствует фиче, то можно описать n фичей. Но если фичи sparse, т.е. встречаются редко, то эта пачка нейронов (n-мерный вектор) может описывать и гораздо больше фичей N>>n. Это достигается за счёт того, что фичи будут образовывать неортогональные вектора. Фичи будуту интерферировать. Но до определенного момента разреженности это некритично. С ростом размерности возможности по упаковке всё большего и большего кол-ва фичей только растут.

По факту можно моделировать ортогональную сеть большой размерности с помощью сетки меньшей размерности. При этом можно кроме компактного хранения можно даже и вычисления проводить непосредственно на этом упаковнном векторном пространстве.

![](./img/2025-11-19/toy-models-superposition-1.jpg)

[РдзбЛогСозн] и [РдзбЛогМышл] активно восставал против "нейрона бабушки", и что мозг оперирует бинарным кодами. Но получается, что суперпозиция в нейронных сетях - вот тебе и бинарные коды.

# [TransCircMathFrmwrkForTransformer] Transformer Circuits. A Mathematical Framework for Transformer Circuits

From: 3BL1BRAttentionTransf  
Source: https://transformer-circuits.pub/2021/framework/index.html#notation-tensor-product    
Read: 2025-11-17  
File: math_frmwrk_for_transf_circuits.mht  

Residual connections - это основной (а не какой-то дополнительные/вторичный) тракт, который используется для обмена информации между attention heads. По факту мы имеем, что каждая голова добавляет свою информацию в этот тракт. Т.к. размерность тракта большая, то там могут существовать области, котрые удобно использовать для обмена. Некоторые головы могут вытирать информацию других голов. 

Короче attention heads в основном передвигают информацию. 

> These models involve two fundamental components – MLP (“multi-layer perceptron”) layers, which process information within each token position using collections of neurons; and attention layers, which move information between token positions. (https://transformer-circuits.pub/2025/attribution-graphs/biology.html#method-overview)


<img src="./img/2025-11-17/math-framework-transformer-1.jpg" width=600 /><br/>
<img src="./img/2025-11-17/math-framework-transformer-2.jpg" width=600 /><img src="./img/2025-11-17/math-framework-transformer-3.jpg" width=600 /><img src="./img/2025-11-17/math-framework-transformer-4.jpg" width=600 />

# [TransCircBiologyLLM] Transfomer Circuits. On the Biology of a Large Language Model

From: 3BL1BRLLMStoreFacts  
Read: 2025-11-18  
Source: https://transformer-circuits.pub/2025/attribution-graphs/biology.html   
File: biology_llm.mht  

- Огромное богатство фичей (типа "Rhymes with  sound “eet”/“it”/“et” или "~36 + ~60" или "putting letters together").
- Абстрактные фичи посередине. Причем многие такие фичи language-agnostic. 
- В более мелких моделях меньше абстрактных фичей!! Почему кошак/собака тупее человека.
- В параллель сразу несколько цепочек бежит.
- Цепочки могут подавлять друг-друга (типа, нельзя говорить вещи, которые могут навредить).
- на примере "Addition" можно увидеть, как цепочки усиливают/подкрепляют друг-друга. Ещё же там видно, что сложение состоит из многих трактов: типа _6 + _9 или "около 40 + около 50".

![](./img/2025-11-18/biology-llm-1.jpg)

# [HintonGlomFinal] Geoffrey Hinton. How to represent part-whole hierarchies in a neural network

From: ГайнцеваСтруктМышлЧел  
Read: 2025-11-15  
Source: https://arxiv.org/abs/2102.12627  

Проблема парсинга визуальных сцен. Чтобы получить что-то типа графа сцены. Но этот граф должен быть специфичным для каждой сцены и отражать смысл происходящего с иерархией (типа лицо->ухо->серёжка). 

Концепция GLOM от "agglomerate":
- много однотипных колонок (The GLOM architecture is composed of a large number of columns which all use exactly the same weights). Точнее даже идентичных, т.к. веса всех сеток одинаковы
- каждая колонка привязана к своему месту (spatial location)
- колонка изучает мир целиком начиная от базового уровня и заканчивая самым высшим
- для описания всего на свете хватит и 5 уровней
- каждый уровень в колонке представлен автоэнкодером
- дискриминация есть динамичный процесс, а не статичный. В ходе этого процесса каждая из колонок с развёрткой по времени (использует инфу с предыдущего этапа) изучает информацию, делает bottom-up предсказания и top-down предсказания. Также колонки могут обмениваться инфой друг с другом.
  
  <img src="./img/2025-11-17/glom-1.jpg" width=600 />
- в конце концов наступает (или не наступает) стабилизация. В результате образуются "островки" одинковых эмбеддингов на разных уровнях колонок. Эти островки соответствуют частям объектов. Этим отсровкам можно сопоставить граф, который мы искали

  <img src="./img/2025-11-17/glom-2.jpg" width=600 />


По описанию колонки - прямая ассоциация с [РдзбЛогСозн] и [РдзбЛогМышл]. У Редозубов колонки тоже живут своей жизнью и познают мир независимо друг от друга во всей полное.

GLOM можно рассматривать, как трансформер.

![](./img/2025-11-17/glom-3.jpg)

> There is strong psychological evidence that people parse visual scenes into part-whole hierarchies and model the viewpoint-invariant spatial relationship between a part and a whole as the coordinate transformation between intrinsic coordinate frames that they assign to the part and the whole
> 
> A more radical version of universal capsules, which avoids both symmetry breaking and routing, is to pre-assign a universal capsule to every location in the image. These ubiquitous universal capsules can be used to represent whatever happens to be at that location. An even more profligate version is to dedicate several different levels of ubiquitous universal capsule to each location so that a location can belong to a scene, an object, a part and a sub-part simultaneously. This paper explores this profligate way of representing the part-whole hierarchy.
>
> One of the motivations for GLOM was the idea that the whole object has a compound color which might be called ”pale-green-or-mauve” and at the object level every location belonging to the object has exactly the same, compound color. The object is pale-green-and-mauve all over.
>
> Work on the use of neural fields for generating images has established that there are much better ways to represent the location than using two scalars for its x and y coordinates [Sitzmann et al., 2020, Mildenhall et al., 2020]. The product of a delta function at the location with both horizontal and vertical sine and cosine waves of various frequencies works well. A similar representation is used in transformers for the position of a word fragment in a sentence.

# [LeCunPathToAutonomMachIntel] Yann LeCun. A Path Towards Autonomous Machine Intelligence

From: ГайнцеваСтруктМышлЧел


# [ГайнцеваСтруктМышлЧел] Татьяна Гайнцева. Структурное мышление или важное отличие человека от ИИ

Read: 2022-12-01
Read: 2025-11-15
Source: https://habr.com/ru/articles/687646/

Человек мыслит структурами и делает это непринужденно. В заметке 60 я тоже до этого дотащился - без абстракций не получится даже побороть проблему фенотипичности. Проблема фенотипичности тоже в статье затрагивается (типа друг купил новую шляпу).

Ссылки на работы Хинтона [HintonGlomFinal] и ЛеКуна [LeCunPathToAutonomMachIntel].

# [MMinskyFrmwrkReprKnowldg] Marvin Minsky. A Framework for Representing Knowledge

Read: 2025-10-31  
From: РдзбЛогСозн

Изучил манускрин Марвина Минского "A Framework for Representing Knowledge". На эту теорию была ссылка в [РдзбЛогСозн] "Логике сознания" Редозубова.

Ключевые моменты:
1) связь с байесовским анализом (БА). В теории фреймов терминали имеют предзаполненное значение - аналог априорного знания в БА
2) одновременно текущая реальность описывается множеством фреймов (см. пример с диагностикой и починкой двигателя в разделе 3.6). Каждый фрейм - это своя точка зрения, свой мир. Связь с контекстами Редозубова
3) иерархичность (вложенность) фреймов: терминали одного фрейма могут быть другими фреймами и т.д.
4) хотя явно нигде не проговаривалось, но из пунктов 2 и 3 следует, что системе приходится работать с очень богатым фича спейсом (cues). Сотни, тысячи, а может и десятки тысяч фичей присутствуют в памяти для описания реальности. Hyper-dimensional computing P. Kanerva?
5) дискретностью переменных в фича спейсе. Возможно даже бинарность. Раздел 5.5.
6) непрерывность изменения. "the system prefers to make small changes whenever possible" (1.6). В каждый момент времени система обновляет небольшую часть фреймов. Это как реклама трамвайчик в московском метро: трамвай едет, а по бокам от него в режиме FIFO появляются новые объекты, а старые исчезают.
7) символичность мыслительных процессов, включая даже seeing.

В целом многие мои мысли подтвердились, прямо принципиального нового ничего не нашлось, но всё равно было занятно. Единственный момент - осталось ощущение deus ex machina. Т.е. понятно, что когда в памяти эти фреймы со всей процедурной обвязкой есть, тогда всё выглядит логично. Вопрос: как эти фреймы образовались в памяти???

Интересные цитаты:
- There is room in the anatomy and genetics of the brain for much more mechanism than anyone today is prepared to propose, and we should concentrate for a while more on sufficiency efficiency than on necessity
- More likely, I would think, one has special systems for important objects but also a variety of frames for generally useful "basic shapes"; these are composed to form frames for new cases.
- Visual experience seems continuous. One reason is that we move continuously.
- The problem is particularly important for fast-moving animals because a model of the scene must be built up from different, partially analyzed views. Perhaps the need to do this, even in a relatively primitive fashion, was a major evolutionary stimulus to develop frame-systems, and later, other symbolic mechanisms.
- The context of the Piaget-Inhelder quotation presents evidence that complete coordination structures of this sort are not available to children in their first decade.
- Seeing seems more vivid than Imagining because its assignments are less flexible; they more firmly resist the attempts of other processes to modify them.
- I was disturbed by how well it explained some things and how poorly others. But it was foolish to expect any single scheme to explain very much about thinking.
- One does not learn well if the required jumps are too large: one cannot really understand animal stories until one possesses the conventional personality frames for the wolf, pig, fox, etc.)
- Here is one way that language affects thinking: each such linguistic convention focuses special attention on filling certain terminals.
- The key words and ideas of a discourse evoke substantial thematic or scenario structures, drawn from memory with rich default assumptions.
- The justification of Napoleon's statement–if, indeed, he ever made it–that those who form a picture of everything are unfit to command, is to be found in the first of these defects.
- Why have two separate frames, rather than one integrated structure to represent the generator? I believe that in such a complex problem one can never cope with many details at once. At each moment one must work within a reasonably simple framework.
- This obsession has kept us from seeing that thinking begins with defective networks that are slowly (if ever) refined and updated.

# [ZKulpaPutOrdImpos] Zenon Kulpa. Putting order in the impossible. 

Source: https://im-possible.info/english/articles/kulpa/putting-order.html  
Read: 2025-11-04  
From: MMinskyFrmwrkReprKnowldg

Perception, 1987, vol. 16, pp. 201-214.  

Наводкой на статью было упоминание невозможной пирамиды Huffman в [MMinskyFrmwrkReprKnowldg]. 

Интересное было - это система символьного описания невозможныъ фигур в виде перечисления типов углов. Ещё одна более продвинутая символьная система для описания брусков. 

Вдохновило на думание в сторону символьной системы описания фичей в проблема распознавания MNIST цифер.

# [MLInsideEigenVectorsML] Зачем они нужны в ML? Собственные значения и собственные векторы

Source: https://www.youtube.com/watch?v=Y_HqYTheDKk  
Read: 2025-10-07  
From: YoutubeRecomm

1) Гугловский PageRank, как задача нахождения собств. векторов для матрицы переходов
2) Спектральная кластеризация

# [GleichPagerankBeyondWeb] David F. Gleich. PageRank beyond the Web

Read: 2025-11-05  
From: MLInsideEigenVectorsML

Решение задачи \$(αP + (1 − α)ve^T)x = x\$, где P - стохастическая матрица ссылок, α- параметр телепортации, v - векотр распределения для телепортации (может быть просто равномерно).  

Оказывается можно смотреть на это как на цепь Маркова и, следовательно, просто итеративно и быстро близко подойти к решению даже для больших размерностей. Хотя ещё более правильно посмотреть на это как на применение метода Power iteration (https://en.wikipedia.org/wiki/Power_iteration). Т.е. способ найти собственный вектор итеративно. Другими словами сходимость цепи Маркова работает по принципу Power method =)

На основе PageRank можно сопоставлять графы (стр. 11) - IsoRank. Примерно такое я пытался тоже сделать, но на основе разницы квадратов/cossim. Но только с ИсоРанком можно графы разного размера сопоставлять.

> Zhou et al. [2003] used a localized PageRank computation to infer the identity of handwritten digits from only a few examples.

# [MBiezenMarkovChains] Michel van Biezen. Probability & Statistics 3 - Markov Chains

Source: https://www.youtube.com/playlist?list=PLX2gX-ftPVXWgcF0WATMDr-AfvfaYjJZ3  
Read: 2025-11-06  
From: GleichPagerankBeyondWeb  

Фокус на итеративном схождении стохастической матрицы P. Явно рассказывает, что есть Column-stochastic, есть Row-stochastic.  
Ещё интересно про Absorbing Markov Chains - когда у некоторых узлов есть только сток.
![](./img/2025-11-06/biezen-1.jpg)

# [ZhouLearnLocGlobConsist] Zhou et al. Learning with Local and Global Consistency

Read: 2025-11-05  
From: GleichPagerankBeyondWeb

Идеи аналогичные ПейджРанк (стохастическая матрица ссылок + телепортация) для semi-supervised (transductive) обучения:
1) представляем точки датасета в виде графа
2) строим матрицу смежности на основе экспоненцированного гаусс ядра
3) потом собираем матрицу графовый Лапласиан 
4) итерируем как в пейджранке: F(t+1) = αSF(t) + (1−α)Y

Пункт 4 - это на самом деле Power method/iteration - https://en.wikipedia.org/wiki/Power_iteration. Т.е. способ найти собственный вектор итеративно. Изначально я думал, что пункт 4 работает по принципу сходимости в цепи Маркова, но теперь думаю, что это сходимость цепи Маркова работает по принципу Power method =)

К сожалению про распознавание цифр совсем чуть-чуть. Можно сказать мимоходом. 

![](./img/2025-11-06/zhou-1.jpg)

# [NgSpectralCluster] Ng et al. On Spectral Clustering: Analysis and an algorithm

Read: 2025-11-06  
From: ZhouLearnLocGlobConsist

Алгоритм спектрального разбиения графа. 

Статья взрастила интерес к спектральной теории графов. Это когда граф описывается матрицей смежности и другими матрицами, а потом анализируются собственные вектора/числа. Вот тут можно продолжить изучение https://fanchung.ucsd.edu/cbms.pdf

Примеры успешных разбиений/кластериаций. Особенно с надписью "NIPS 2001" хорошо вышло.

![](./img/2025-11-06/ng-1.jpg)

# [StfrdMineMassDatst] Stanford. Mining Massive Datasets Online Course

Source: 
- https://www.youtube.com/watch?v=FRZvgNvALJ4
- https://www.youtube.com/watch?v=RJtCR3h9mXQ
- https://www.youtube.com/watch?v=Cedjf9G0otE
- https://www.youtube.com/watch?v=siCPjpUtE0A
- https://www.youtube.com/watch?v=uxsDKhZHDcc


Read: 2025-11-06  
From: NgSpectralCluster  

<img src="./img/2025-11-06/mining-massive-datasets-0.jpg" width=500/> <img src="./img/2025-11-06/mining-massive-datasets-1.jpg" width=500/> <img src="./img/2025-11-06/mining-massive-datasets-2.jpg" width=500/>

# [MBiezenEigen] Michel van Biezen. Linear Algebra: Ch 3 - Eigenvalues and Eigenvectors

Source: 
- https://www.youtube.com/playlist?list=PLX2gX-ftPVXWjzkAjEu4Sd16R4wz55Bwe
- http://www.ilectureonline.com/lectures/subject/MATH/36/331

Read: 2025-11-06  
From: RandomSurf 

Тут подробно про Power method - итеративный способ вычисления собственных векторов (https://en.wikipedia.org/wiki/Power_iteration). 

# [AntDamasioDescartesError] Antonio Damasio. Descartes Error: Emotion, Reason, and the Human Brain.

Antonio studied people with brain damage where emotions generated. These people couldn't make decisions. They could describe what they should do in logic terms, but they found it impossible to make even the simplest choice